# Sample SuperStore - Business Analysis Analytics

This notebook documents a business-analysis engagement to identify and act on low sales in the most recently completed month. The analysis uses `Orders.csv`; as of 7 September 2026, the last completed month is August 2026.


In [6]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

BASE = Path.cwd()
orders_path = BASE.parent / "Orders.csv"
if not orders_path.exists():
    orders_path = Path(r"C:\Users\dhpur\Desktop\SSS_hugging face\Orders.csv")
orders = pd.read_csv(orders_path)
orders["Order Date"] = pd.to_datetime(orders["Order Date"], format="%d-%m-%Y")
for column in ["Sales", "Quantity", "Discount", "Profit"]:
    orders[column] = pd.to_numeric(orders[column])
analysis_month = pd.Period("2026-08", freq="M")
month_orders = orders[orders["Order Date"].dt.to_period("M") == analysis_month].copy()
group_cols = ["Region", "State/Province", "City", "Category", "Sub-Category", "Product ID"]
sales_summary = (month_orders.groupby(group_cols, as_index=False)
    .agg(sales=("Sales", "sum"), quantity=("Quantity", "sum"), discount=("Discount", "mean"), profit=("Profit", "sum")))
median_sales = sales_summary["sales"].median()
bottom_50 = sales_summary[sales_summary["sales"] <= median_sales].copy()
lowest = bottom_50.sort_values(["sales"] + group_cols, kind="stable").iloc[0]
discount_text = f"{lowest['discount']:.0%}"
profit_reason = (f"Profit is negative ({lowest['profit']:.2f}), indicating an unprofitable sales mix."
                 if lowest["profit"] < 0 else
                 f"Profit is only {lowest['profit']:.2f} for the recorded sales, limiting contribution.")
lower_sales_row = pd.DataFrame([{
    "region": lowest["Region"], "state": lowest["State/Province"], "city": lowest["City"],
    "category": lowest["Category"], "subCategory": lowest["Sub-Category"], "productid": lowest["Product ID"],
    "why_1": f"Sales of {lowest['sales']:.2f} are the lowest observation in the bottom 50% for {analysis_month}.",
    "why_2": f"Only {int(lowest['quantity'])} unit(s) were sold in the period, indicating weak demand.",
    "why_3": f"The average discount was {discount_text}, which may be reducing realized sales value.",
    "why_4": profit_reason,
    "why_5": f"The weakness is concentrated in {lowest['City']}, {lowest['State/Province']} for {lowest['Sub-Category']} in the {lowest['Region']} region."
}])
display(Markdown(f"**Analysis month:** {analysis_month}  |  **Orders:** {len(month_orders):,}  |  **Bottom-50% threshold:** ${median_sales:,.2f}"))
display(lower_sales_row)


**Analysis month:** 2026-08  |  **Orders:** 223  |  **Bottom-50% threshold:** $73.29

,region,state,city,category,subCategory,productid,why_1,why_2,why_3,why_4,why_5
0,South,Tennessee,Memphis,Technology,Accessories,TEC-AC-10003709,Sales of 1.58 are the lowest observation in th...,"Only 2 unit(s) were sold in the period, indica...","The average discount was 20%, which may be red...","Profit is only 0.48 for the recorded sales, li...","The weakness is concentrated in Memphis, Tenne..."


## Step 1 - Assessment

### Situation Statement
August 2026 sales contain products and locations in the bottom half of the sales distribution. Leaders need a repeatable, evidence-based view of the lowest-sales opportunity so they can prioritize investigation and corrective action.

### Ishikawa / Fishbone Diagram
Suspected low-sales causes: **People** (limited product knowledge or coverage), **Process** (weak replenishment and promotion processes), **Product** (low fit, assortment, or availability), **Price** (discount strategy or competitiveness), **Place** (regional demand or channel coverage), and **Data/Measurement** (late, incomplete, or inconsistent signals).

### 5 WHYs
1. Why are sales low? The selected product/location recorded the lowest sales in the bottom 50%.
2. Why did it record low sales? Only a small number of units were purchased during the month.
3. Why were units low? Customer demand, visibility, or availability may be insufficient in the location.
4. Why might demand or visibility be insufficient? Local assortment, promotion, price, or sales coverage may not match the market.
5. Why is the mismatch not corrected? There is no regular low-sales alert with assigned owners, root-cause validation, and tracked actions.

### Feasibility Study
- **Technical:** Feasible; the CSV has the required sales, date, product, category, and location fields.
- **Operational:** Feasible; a monthly report can be reviewed by sales and merchandising owners.
- **Economic:** Feasible; existing data and low-cost Python/spreadsheet automation are sufficient.
- **Schedule:** Feasible; an initial report can be produced in one sprint and refined after feedback.
- **Risks:** Reasons are hypotheses until validated with operational evidence.

### BRD - Business Requirement Diagram
`Orders.csv -> Validate dates/metrics -> Aggregate monthly sales -> Calculate median -> Filter bottom 50% -> Select one lowest-sales row -> Add five WHY hypotheses -> Export Excel/report -> Review and act`

### SWOT Analysis
| Strengths | Weaknesses |
|---|---|
| Existing transaction data; repeatable metric; clear output | Reasons require validation; batch CSV; no stock-out field |

| Opportunities | Threats |
|---|---|
| Alerts, targeted promotions, assortment optimization | Seasonality, data quality, over-discounting, competitors |


## Step 2 - Identifying Stakeholders

### RACI Matrix
| Activity | Executive Sponsor | Sales Lead | Merchandising | Data/BA | Finance | IT |
|---|---|---|---|---|---|---|
| Approve objective | A | C | C | R | C | I |
| Define metric | I | C | C | R/A | C | I |
| Validate causes | I | R/A | R | C | C | I |
| Approve action | A | R | R | C | C | I |
| Publish report | I | C | C | R/A | I | C |
| Transition support | I | R | C | C | I | A |

`R` Responsible, `A` Accountable, `C` Consulted, `I` Informed.

### Stakeholder Map - Matrix and Power/Interest Grid
| Stakeholder | Power | Interest | Engagement |
|---|---|---|---|
| Executive Sponsor | High | Medium | Manage expectations |
| Sales Lead | High | High | Manage closely |
| Merchandising | Medium | High | Collaborate |
| Data/BA | Medium | High | Deliver and facilitate |
| Finance | High | Medium | Consult and validate |
| IT/Data Platform | Medium | Medium | Keep satisfied |
| Regional Managers | Low | High | Keep informed and involve |

High power/high interest: Sales Lead, Sponsor, Merchandising. High power/low interest: Finance, IT. Low power/high interest: Regional Managers. Low power/low interest: monitor.

### Onion Map
`Core:` BA, Sales Lead, Merchandising.  
`Inner ring:` Executive Sponsor, Finance, IT/Data Platform.  
`Outer ring:` Regional Managers, Customer teams, suppliers, end customers.

### Engagement Matrix
| Group | Current state | Desired state | Method |
|---|---|---|---|
| Decision makers | Aware | Supportive | Steering reviews |
| Operational owners | Mixed | Actively engaged | Workshops and action register |
| Data/technology | Consulted | Committed | Data contract and release checklist |
| Affected teams | Unaware | Informed and heard | Demos, surveys, transition training |


## Step 3 - Business Case

### Situation Statement
A single lowest-sales opportunity is surfaced from the bottom 50% so limited commercial capacity can focus on the most actionable product/location combination.

### Gap Analysis
| Current state | Desired state | Gap | Action |
|---|---|---|---|
| Manual low-sales review | Automated repeatable alert | No standard workflow | Schedule notebook/report |
| Reasons are implicit | Reasons recorded and validated | No root-cause ownership | Interviews and reason fields |
| Actions not traced to outcomes | Owned actions with measures | Limited accountability | RTM, Kanban, scorecard |

### Cost-Benefit Analysis
| Cost | Benefit |
|---|---|
| BA and stakeholder time | Faster prioritization and less analysis effort |
| Automation maintenance | Repeatable monthly monitoring |
| Targeted promotions/assortment changes | Potential sales and margin recovery |
| Training and transition | Sustained adoption |

### Value Chain Analysis
`Data capture -> Data quality -> Sales aggregation -> Opportunity prioritization -> Root-cause validation -> Commercial action -> Outcome measurement`.


## Step 4 - Planning the Project

### Scope of Work
**In scope:** monthly extraction, bottom-50% logic, one-row prioritization, five WHY hypotheses, Excel output, validation, traceability, UAT, and transition. **Out of scope:** automatic price changes, inventory execution, causal proof without additional data, and unapproved production integration.

### Methodology
Use an iterative BABOK-aligned approach: discover, analyze, validate, prototype, test, release, and measure. Each sprint ends with review and a decision log.

### Stakeholders
Executive Sponsor, Sales Lead, Merchandising, Finance, Data/BA, IT/Data Platform, and Regional Managers.

### Schedule
| Sprint | Focus | Exit criterion |
|---|---|---|
| 1 | Assessment and profiling | Scope and metric approved |
| 2 | Elicitation and BRD | Requirements baselined |
| 3 | Prototype, RTM, business case | Prototype accepted |
| 4 | Build, verification, UAT | UAT sign-off |
| 5 | Release and benefits review | Handover complete |

### Requirements Management
Maintain versioned requirement ID, source, priority, owner, acceptance criteria, status, dependency, and change history. Baseline after sponsor approval.

### Deliverables
Notebook, validated Excel report, BRD, stakeholder artifacts, RTM, test evidence, UAT sign-off, release roadmap, transition pack, and lessons-learned log.

### Communication Plan
| Audience | Cadence | Channel | Owner |
|---|---|---|---|
| Sponsor | Sprint review | Steering meeting | BA |
| Sales/Merchandising | Weekly | Workshop/dashboard | Sales Lead |
| Data/IT | Twice weekly | Working session | Data/BA |
| Wider users | Release/training | Email/demo | Change owner |

### BA Work Assessment
Measure requirement completeness, stakeholder participation, traceability coverage, defect leakage, UAT acceptance, time saved, and evidence of an owned corrective action.


## Step 5 - Determining Requirements: BABOK 9 Elicitation Techniques
1. **Brainstorming:** generate hypotheses for demand, price, product, location, and process causes.
2. **Document Analysis:** inspect Orders.csv, prior reports, promotions, and inventory documents.
3. **Focus Groups:** gather regional and merchandising perspectives.
4. **Interface Analysis:** define inputs/outputs between CSV, notebook, Excel, BI, and action tracking.
5. **Interviews:** confirm definitions, ownership, and reason validation.
6. **Observation:** observe how managers identify and respond to low sales.
7. **Prototyping:** demonstrate the one-row output and reason fields.
8. **Requirements Workshops:** agree scope, threshold, acceptance criteria, and change rules.
9. **Survey/Questionnaire:** collect feedback on usability, cadence, and actionability.

### Initial Functional Requirements
| ID | Requirement | Priority | Acceptance criterion |
|---|---|---|---|
| FR-01 | Load and validate Orders.csv | Must | Invalid dates/metrics surface |
| FR-02 | Analyze last completed month | Must | Month is explicit and reproducible |
| FR-03 | Calculate bottom 50% sales | Must | Median threshold shown |
| FR-04 | Return exactly one lowest-sales row | Must | Output has one data row |
| FR-05 | Include six dimensions and five WHY columns | Must | All 11 columns populated |
| FR-06 | Export to Excel | Must | Workbook opens with schema |


## Step 6 - Traceability and Monitoring Requirements

### Requirements Traceability Matrix (RTM)
| Req ID | Source | Design/Artifact | Test | Status |
|---|---|---|---|---|
| FR-01 | Sales Lead | Data-loading cell | T-01 | Planned |
| FR-02 | Sponsor | Analysis-month parameter | T-02 | Planned |
| FR-03 | Merchandising | Median/bottom-50 logic | T-03 | Planned |
| FR-04 | Sponsor | Lowest-row selection | T-04 | Planned |
| FR-05 | Sales/BA | Output schema and WHYs | T-05 | Planned |
| FR-06 | Users | Excel export | T-06 | Planned |

### Change Control Board
The CCB consists of the Executive Sponsor, Sales Lead, Merchandising, Finance, and Data/BA. Every change records request, reason, impact, cost, priority, decision, and approver.

### Task Board / Kanban
`Backlog -> Ready -> In Progress -> Review -> UAT -> Done`.

| Work item | Owner | State | Definition of done |
|---|---|---|---|
| Data validation | Data/BA | Done for prototype | Checks pass |
| Low-sales logic | Data/BA | In Progress | Threshold and tie-break documented |
| Cause validation | Sales/Merchandising | Ready | WHYs confirmed or revised |
| UAT | Sales Lead | Backlog | Signed acceptance evidence |


## Step 7 - Evaluating the Solution

### Testing & Verification
Verify file existence, required columns, date/numeric parsing, month selection, median, bottom-half filter, deterministic tie-break, row count, Excel schema, and non-empty WHY fields. Reconcile the selected row to a direct grouped-sales calculation.

### User Acceptance Testing (UAT)
| Scenario | Expected result | Owner |
|---|---|---|
| Run with August 2026 data | One lowest-sales row returned | Sales Lead |
| Inspect dimensions | Values match source | Merchandising |
| Review WHYs | Five useful hypotheses | Regional Manager |
| Open workbook | Columns and values usable | Sponsor |
| Repeat next month | Month updates without code changes | Data/BA |

### Balanced Scorecard
| Perspective | Objective | Measure | Target |
|---|---|---|---|
| Financial | Improve contribution | Sales/profit of selected item | Positive trend |
| Customer | Improve local relevance | Demand/availability feedback | Monthly review |
| Internal process | Reduce identification time | Report cycle time | Same business day |
| Learning and growth | Build adoption | UAT/adoption rate | 90% of target users |


## Step 9 - Preparing for Release and Transition

### Release Roadmap
1. Prototype: notebook and one-row output.
2. Pilot: validate with Sales and Merchandising for one cycle.
3. Controlled release: publish report with owner and review cadence.
4. Scale: automate scheduling, add inventory/promotion data, and expand monitoring.

### Transition Planning Framework
| Area | Transition action |
|---|---|
| People | Train Sales, Merchandising, and Regional Managers |
| Process | Publish monthly runbook and escalation path |
| Technology | Store notebook, input, output, and version metadata |
| Data | Define schema checks, refresh timing, and retention |
| Support | Assign business and Data/IT support |
| Governance | Review access, CCB changes, and benefits monthly |

### Lessons Learned (Kaizen)
- Start with one decision-ready row to reduce noise and increase ownership.
- Make the analysis month and threshold visible to prevent ambiguity.
- Treat WHYs as hypotheses until validated with operational evidence.
- Add inventory, stock-out, promotion, and competitor data next.
- Review the scorecard monthly and convert lessons into backlog items.

### Conclusion
The notebook provides an auditable BA lifecycle from assessment through transition, while the executable analysis produces the requested single lowest-sales observation for the last completed month.
